# Data control panel

Local control panel for checking data freshness, updating the tournament master file, and running the project smoke check.

In [ ]:
from __future__ import annotations

import subprocess
import sys
from pathlib import Path

from IPython.display import Markdown, display
import ipywidgets as widgets


def find_project_root(start: Path | None = None) -> Path:
    start = Path.cwd() if start is None else Path(start).resolve()
    for candidate in [start, *start.parents]:
        if (candidate / '.git').exists() and (candidate / 'README.md').exists():
            return candidate
    raise FileNotFoundError('Could not find project root.')


PROJECT_ROOT = find_project_root()
PYTHON = sys.executable


def run_project_command(args: list[str]) -> str:
    completed = subprocess.run(
        [PYTHON, *args],
        cwd=PROJECT_ROOT,
        capture_output=True,
        text=True,
        encoding='utf-8',
        errors='replace',
    )
    output = completed.stdout.strip()
    if completed.stderr.strip():
        output = f'{output}\n\nSTDERR:\n{completed.stderr.strip()}'.strip()
    if completed.returncode != 0:
        raise RuntimeError(output or f'Command failed with code {completed.returncode}')
    return output


status_button = widgets.Button(description='Refresh status', button_style='info')
tournaments_button = widgets.Button(description='Update tournaments master', button_style='warning')
verify_button = widgets.Button(description='Verify project', button_style='success')
output = widgets.Output(layout={'border': '1px solid #ddd', 'padding': '8px'})


def show_command(title: str, args: list[str]) -> None:
    with output:
        output.clear_output()
        display(Markdown(f'### {title}'))
        try:
            result = run_project_command(args)
            print(result)
        except Exception as exc:
            print(exc)


status_button.on_click(lambda _: show_command('Data status', ['scripts/data_status.py', '--write-manifest']))
tournaments_button.on_click(lambda _: show_command('Tournament master update', ['scripts/bootstrap_tournaments_master.py']))
verify_button.on_click(lambda _: show_command('Project verification', ['scripts/verify_project.py']))

display(Markdown(f'Project root: `{PROJECT_ROOT}`'))
display(widgets.HBox([status_button, tournaments_button, verify_button]))
display(output)
show_command('Data status', ['scripts/data_status.py', '--write-manifest'])
